# K-Nearest Neighbors (KNN) - Clasificación No Paramétrica

Bienvenido al tercer notebook de algoritmos supervisados. K-Nearest Neighbors es un algoritmo **no paramétrico** que utiliza la similitud entre puntos para hacer predicciones.

Al finalizar este notebook, serás capaz de:

* Comprender la filosofía de los algoritmos basados en instancias
* Implementar KNN desde cero para clasificación y regresión
* Calcular diferentes métricas de distancia (Euclidiana, Manhattan)
* Entender el impacto del hiperparámetro K en el rendimiento
* Aplicar validación cruzada para seleccionar K óptimo
* Reconocer cuándo normalizar features es crítico
* Evaluar ventajas y desventajas de KNN vs algoritmos paramétricos

**¿Por qué K-Nearest Neighbors?**

1. **Simplicidad conceptual**: Fácil de entender e implementar
2. **No paramétrico**: No hace suposiciones sobre la distribución de datos
3. **Flexible**: Funciona para clasificación y regresión
4. **Fronteras de decisión complejas**: Puede capturar patrones no lineales
5. **Baseline útil**: Excelente para comparar con otros algoritmos

**Diferencias clave con algoritmos anteriores:**
- **Regresión Lineal/Logística**: Aprenden parámetros ($\mathbf{w}$, $b$) durante entrenamiento
- **KNN**: No aprende parámetros, guarda todos los datos de entrenamiento (lazy learning)

## Nota Importante sobre los Ejercicios

Antes de comenzar con los ejercicios, ten en cuenta lo siguiente:

1. NO agregues declaraciones `print` adicionales en las funciones graduadas
2. NO agregues celdas de código adicionales entre los ejercicios
3. NO cambies los parámetros de las funciones
4. Implementa usando NumPy (operaciones vectorizadas cuando sea posible)
5. NO cambies el código de las pruebas automáticas

Si experimentas errores al ejecutar las pruebas, primero verifica estos puntos antes de buscar ayuda.

<a name='1'></a>
## Tabla de Contenidos
- [1 - Paquetes](#1)
- [2 - Teoría de K-Nearest Neighbors](#2)
- [3 - Implementación desde Cero](#3)
  - [Ejercicio 1](#ex01)
- [4 - Clasificación con KNN](#4)
- [5 - Selección del Hiperparámetro K](#5)
  - [Ejercicio 2](#ex02)
- [6 - Referencias](#6)

<a name='1'></a>
## 1 - Paquetes

Ejecuta la siguiente celda para importar los paquetes que usarás en este notebook:

* **NumPy**: Operaciones numéricas y cálculo de distancias
* **Matplotlib**: Visualización de resultados
* **Collections**: Counter para votación de clases
* **Scikit-learn**: Generación de datos sintéticos
* **Testing utilities**: Verificación automática de ejercicios

In [ ]:
# ==========================================
# CONFIGURACIÓN DEL ENTORNO
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import sys
from pathlib import Path
from sklearn.datasets import make_classification

# Configurar matplotlib inline
%matplotlib inline

# Agregar el directorio raíz al path
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Importar utilidades de testing
from utils.testing_utils import print_success, print_error, print_info
from tests.supervisados.test_03_knn import (
    test_ejercicio_1_distancia,
    test_ejercicio_2_knn_predict
)

print("✅ Paquetes importados correctamente")
print(f"📦 NumPy version: {np.__version__}")

<a name='2'></a>
## 2 - Teoría de K-Nearest Neighbors

KNN es un algoritmo de **aprendizaje basado en instancias** (instance-based learning) o **lazy learning**, porque no construye un modelo explícito durante el entrenamiento.

### 2.1 - Algoritmo de KNN

El algoritmo para predecir la clase de un nuevo punto $\mathbf{x}_{\text{nuevo}}$ es:

**Paso 1:** Calcular la distancia entre $\mathbf{x}_{\text{nuevo}}$ y todos los puntos del conjunto de entrenamiento

**Paso 2:** Seleccionar los $K$ puntos más cercanos (vecinos)

**Paso 3:** 
- **Para clasificación**: Asignar la clase más común entre los $K$ vecinos (votación por mayoría)
- **Para regresión**: Asignar el promedio de los valores de los $K$ vecinos

### 2.2 - Métricas de Distancia

**Distancia Euclidiana** (la más común):

$$d_{\text{Euclid}}(\mathbf{x}_1, \mathbf{x}_2) = \sqrt{\sum_{i=1}^{n} (x_{1i} - x_{2i})^2} = \|\mathbf{x}_1 - \mathbf{x}_2\|_2$$

**Distancia Manhattan** (City Block):

$$d_{\text{Manhattan}}(\mathbf{x}_1, \mathbf{x}_2) = \sum_{i=1}^{n} |x_{1i} - x_{2i}| = \|\mathbf{x}_1 - \mathbf{x}_2\|_1$$

**Distancia de Minkowski** (generalización):

$$d_{\text{Minkowski}}(\mathbf{x}_1, \mathbf{x}_2) = \left(\sum_{i=1}^{n} |x_{1i} - x_{2i}|^p\right)^{1/p}$$

donde $p=1$ es Manhattan, $p=2$ es Euclidiana, y $p \to \infty$ es Chebyshev.

### 2.3 - Votación para Clasificación

Para $K$ vecinos con clases $\{c_1, c_2, ..., c_K\}$, la predicción es:

$$\hat{y} = \text{argmax}_{c} \sum_{i=1}^{K} \mathbb{1}(c_i = c)$$

donde $\mathbb{1}$ es la función indicadora (1 si verdadero, 0 si falso).

**Variante ponderada**: Asignar mayor peso a vecinos más cercanos:

$$\hat{y} = \text{argmax}_{c} \sum_{i=1}^{K} \frac{1}{d_i + \epsilon} \cdot \mathbb{1}(c_i = c)$$

donde $d_i$ es la distancia al vecino $i$ y $\epsilon$ es un pequeño valor para evitar división por cero.

### 2.4 - El Hiperparámetro K

El valor de $K$ controla el balance entre bias y varianza:

- **K pequeño** (ej. K=1): 
  - Baja bias, alta varianza
  - Frontera de decisión muy flexible
  - Sensible a outliers y ruido
  
- **K grande** (ej. K=N):
  - Alta bias, baja varianza
  - Frontera de decisión más suave
  - Puede ignorar patrones locales

**Regla práctica**: $K = \sqrt{N}$ donde $N$ es el número de muestras de entrenamiento.

<a name='3'></a>
## 3 - Implementación desde Cero

Vamos a implementar KNN paso a paso, comenzando con el clasificador.

### 3.1 - Estructura de la Clase

Nuestra clase `KNNClassifier` tendrá:
* `fit(X, y)`: Guarda los datos de entrenamiento (no aprende parámetros)
* `_euclidean_distance(x1, x2)`: Calcula distancia entre dos puntos
* `predict(X)`: Predice clases para múltiples puntos
* `_predict_single(x)`: Predice clase para un único punto
* `score(X, y)`: Calcula accuracy

### 3.2 - Código de Implementación

In [ ]:
class KNNClassifier:
    """
    K-Nearest Neighbors Classifier implementado desde cero.
    
    Parámetros:
    -----------
    k : int
        Número de vecinos a considerar
    """
    
    def __init__(self, k=3):
        self.k = k
        self.X_train = None
        self.y_train = None
    
    def fit(self, X, y):
        """
        'Entrena' el modelo guardando los datos.
        KNN es lazy learning - no aprende parámetros.
        
        Parámetros:
        -----------
        X : array-like, shape (n_samples, n_features)
            Features de entrenamiento
        y : array-like, shape (n_samples,)
            Target de entrenamiento
        """
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        return self
    
    def _euclidean_distance(self, x1, x2):
        """Calcula distancia euclidiana entre dos puntos"""
        return np.sqrt(np.sum((x1 - x2) ** 2))
    
    def predict(self, X):
        """
        Predice clases para X.
        
        Parámetros:
        -----------
        X : array-like, shape (n_samples, n_features)
            Features para predecir
        
        Returns:
        --------
        array, shape (n_samples,)
            Clases predichas
        """
        X = np.array(X)
        if len(X.shape) == 1:
            X = X.reshape(1, -1)
        
        predictions = [self._predict_single(x) for x in X]
        return np.array(predictions)
    
    def _predict_single(self, x):
        """Predice clase para un único punto"""
        # Paso 1: Calcular distancias a todos los puntos de entrenamiento
        distances = [self._euclidean_distance(x, x_train) 
                    for x_train in self.X_train]
        
        # Paso 2: Obtener índices de los K vecinos más cercanos
        k_indices = np.argsort(distances)[:self.k]
        
        # Paso 3: Obtener las etiquetas de los K vecinos
        k_nearest_labels = self.y_train[k_indices]
        
        # Paso 4: Votar por la clase más común
        most_common = Counter(k_nearest_labels).most_common(1)
        return most_common[0][0]
    
    def score(self, X, y):
        """Calcula accuracy"""
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

print("✅ Clase KNNClassifier definida correctamente")

<a name='4'></a>
## 4 - Clasificación con KNN

Vamos a aplicar KNN a un problema de clasificación binaria.

### 4.1 - Generar Datos Sintéticos

Creamos un dataset de clasificación balanceado:

In [ ]:
# Generar datos sintéticos
X, y = make_classification(n_samples=100, n_features=2, n_redundant=0,
                          n_informative=2, n_clusters_per_class=1, 
                          random_state=42)

# Visualizar
plt.figure(figsize=(10, 6))
plt.scatter(X[y==0, 0], X[y==0, 1], c='red', label='Clase 0', edgecolors='k', alpha=0.7)
plt.scatter(X[y==1, 0], X[y==1, 1], c='blue', label='Clase 1', edgecolors='k', alpha=0.7)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Datos de Clasificación')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Total de muestras: {len(X)}")
print(f"Features: {X.shape[1]}")
print(f"Clases: {np.unique(y)}")

### 4.2 - División Train/Test

Dividimos los datos para evaluar la capacidad de generalización:

In [ ]:
# Train/Test split
def train_test_split(X, y, test_size=0.2, random_state=42):
    np.random.seed(random_state)
    n = len(X)
    indices = np.random.permutation(n)
    test_size_n = int(n * test_size)
    test_idx = indices[:test_size_n]
    train_idx = indices[test_size_n:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

X_train, X_test, y_train, y_test = train_test_split(X, y)

print(f"Train set: {len(X_train)} muestras")
print(f"Test set: {len(X_test)} muestras")

### 4.3 - Entrenar y Evaluar el Modelo

Recuerda: KNN no "entrena" en el sentido tradicional, solo guarda los datos:

In [ ]:
# Entrenar KNN
knn = KNNClassifier(k=5)
knn.fit(X_train, y_train)

# Evaluar
train_acc = knn.score(X_train, y_train)
test_acc = knn.score(X_test, y_test)

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

if train_acc > test_acc + 0.1:
    print("\n⚠️  Posible overfitting: considera aumentar K")
else:
    print("\n✅ El modelo generaliza bien")

### 4.4 - Visualizar Frontera de Decisión

KNN puede crear fronteras de decisión complejas y no lineales:

In [ ]:
# Visualizar frontera de decisión
def plot_decision_boundary_knn(X, y, model, title="KNN - Frontera de Decisión"):
    """Grafica la frontera de decisión para KNN"""
    # Crear grid
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    
    # Predecir para cada punto del grid
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Graficar
    plt.figure(figsize=(10, 6))
    plt.contourf(xx, yy, Z, alpha=0.4, cmap='RdBu')
    plt.scatter(X[y==0, 0], X[y==0, 1], c='red', label='Clase 0', edgecolors='k', s=50)
    plt.scatter(X[y==1, 0], X[y==1, 1], c='blue', label='Clase 1', edgecolors='k', s=50)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

plot_decision_boundary_knn(X_train, y_train, knn, 
                           title=f"KNN (K={knn.k}) - Frontera de Decisión")

<a name='6'></a>
## 6 - Referencias

### Papers Fundamentales

1. **Fix, E., & Hodges, J. L.** (1951). "Discriminatory analysis. Nonparametric discrimination: Consistency properties." *USAF School of Aviation Medicine*, Randolph Field, Texas.
   - Introducción original del algoritmo KNN

2. **Cover, T., & Hart, P.** (1967). "Nearest neighbor pattern classification." *IEEE Transactions on Information Theory*, 13(1), 21-27.
   - Análisis teórico de KNN y tasa de error

3. **Hastie, T., & Tibshirani, R.** (1996). "Discriminant adaptive nearest neighbor classification." *IEEE Transactions on Pattern Analysis and Machine Intelligence*, 18(6), 607-616.
   - Variante adaptativa de KNN

### Recursos Adicionales

4. **Bishop, C. M.** (2006). *Pattern Recognition and Machine Learning*. Springer. Section 2.5: Nearest-neighbor methods.
   - Tratamiento teórico riguroso

5. **Murphy, K. P.** (2012). *Machine Learning: A Probabilistic Perspective*. MIT Press. Chapter 1.4: KNN classifiers.
   - Perspectiva probabilística

6. **James, G., Witten, D., Hastie, T., & Tibshirani, R.** (2013). *An Introduction to Statistical Learning*. Springer. Section 2.2: Assessing Model Accuracy.
   - KNN como ejemplo de bias-variance tradeoff

### Estructuras de Datos para Aceleración

7. **Bentley, J. L.** (1975). "Multidimensional binary search trees used for associative searching." *Communications of the ACM*, 18(9), 509-517.
   - KD-Trees para búsqueda eficiente

8. **Omohundro, S. M.** (1989). "Five balltree construction algorithms." *International Computer Science Institute Technical Report*.
   - Ball Trees como alternativa a KD-Trees

9. **Muja, M., & Lowe, D. G.** (2009). "Fast approximate nearest neighbors with automatic algorithm configuration." *VISSAPP*, 2(331-340), 2.
   - Algoritmos de approximate nearest neighbors (FLANN library)

### Recursos Online

10. **Scikit-learn Documentation: Nearest Neighbors**
    - https://scikit-learn.org/stable/modules/neighbors.html
    - Implementaciones optimizadas con KD-Tree, Ball Tree, Brute Force

11. **StatQuest: K-nearest neighbors**
    - https://www.youtube.com/watch?v=HVXime0nQeI
    - Explicación visual intuitiva

12. **Coursera: Machine Learning by Andrew Ng**
    - Sección sobre algoritmos no paramétricos

### Variantes y Extensiones

13. **Weighted KNN**: Ponderar vecinos por inverso de distancia
14. **Adaptive KNN**: K diferente para cada punto
15. **KNN Regressor**: Promediar valores en lugar de votar
16. **Local Outlier Factor (LOF)**: Detección de anomalías basada en KNN
17. **SMOTE**: Synthetic Minority Over-sampling usando KNN

### Optimizaciones

18. **Approximate Nearest Neighbors (ANN)**: Annoy, FAISS, NMSLIB
19. **Dimensionality Reduction**: PCA antes de KNN para reducir features
20. **Feature Weighting**: Asignar pesos diferentes a cada feature

## 📘 Resumen y Aplicaciones en ML

<div style="background-color: #e7f3fe; padding: 20px; border-left: 6px solid #2196F3; margin: 20px 0;">

**Conceptos Clave Aprendidos:**

1. **Lazy Learning**: KNN no aprende parámetros, guarda todos los datos de entrenamiento
2. **Instance-Based**: Las predicciones se basan en similitud con instancias conocidas
3. **Hiperparámetro K**: Controla el balance entre bias y varianza
4. **Métricas de Distancia**: Euclidiana es común, pero Manhattan y otras tienen sus usos
5. **No Lineal**: Puede capturar fronteras de decisión complejas sin transformaciones

**Ventajas de KNN:**

✅ **Simple de entender e implementar**: No requiere matemáticas complejas  
✅ **No paramétrico**: No hace suposiciones sobre distribución de datos  
✅ **Versátil**: Funciona para clasificación y regresión  
✅ **Adaptable**: Se adapta automáticamente a nuevos datos  
✅ **Baseline poderoso**: Excelente para comparaciones

**Desventajas de KNN:**

❌ **Predicción lenta**: Debe calcular distancias a todos los puntos  
❌ **Requiere mucha memoria**: Guarda todos los datos de entrenamiento  
❌ **Sensible a escala**: Features con rangos grandes dominan la distancia  
❌ **Curse of dimensionality**: Performance degrada en dimensiones altas  
❌ **Sensible a outliers**: Especialmente con K pequeño

**Aplicaciones Reales:**

- **Sistemas de recomendación**: "Usuarios similares también compraron..."
- **Reconocimiento de patrones**: Clasificación de imágenes, caracteres
- **Detección de anomalías**: Puntos con vecinos lejanos son outliers
- **Imputación de valores faltantes**: Usar promedio de vecinos
- **Compresión de datos**: Representar puntos por sus vecinos

**Optimizaciones para Producción:**

1. **KD-Trees o Ball Trees**: Estructuras de datos para búsqueda rápida
2. **Approximate Nearest Neighbors (ANN)**: Trade-off precisión por velocidad
3. **Feature selection**: Eliminar features irrelevantes
4. **Normalización**: CRÍTICO - siempre normalizar features
5. **Distance weighting**: Ponderar vecinos por inverso de distancia

**Cuándo usar KNN:**

✅ **Dataset pequeño a mediano** (< 100k muestras)  
✅ **Pocas dimensiones** (< 20 features)  
✅ **Necesitas interpretabilidad** (puedes mostrar vecinos)  
✅ **Baseline rápido** antes de modelos complejos  
✅ **Datos limpios** sin muchos outliers

❌ **NO usar con:**
- Datasets grandes (millones de muestras)
- Alta dimensionalidad (cientos de features)
- Requisitos de predicción en tiempo real
- Datos con muchos outliers

</div>

In [ ]:
# GRADED FUNCTION: knn_predict

def knn_predict(x, X_train, y_train, k=3):
    """
    Predice la clase de un punto usando K-Nearest Neighbors.
    
    Parámetros
    ----------
    x : ndarray
        Punto a clasificar, forma (n_features,)
    X_train : ndarray
        Datos de entrenamiento, forma (n_samples, n_features)
    y_train : ndarray
        Etiquetas de entrenamiento, forma (n_samples,)
    k : int
        Número de vecinos a considerar
    
    Retorna
    -------
    int
        Clase predicha
    
    Ejemplo
    -------
    >>> X_train = np.array([[1, 2], [2, 3], [3, 1], [6, 5], [7, 7], [8, 6]])
    >>> y_train = np.array([0, 0, 0, 1, 1, 1])
    >>> x = np.array([3, 3])
    >>> knn_predict(x, X_train, y_train, k=3)
    0
    """
    
    ### YOUR CODE STARTS HERE ###
    # Paso 1: Calcular distancias (puedes usar compute_distances o implementarlo aquí)
    distances = None
    
    # Paso 2: Obtener índices de los K vecinos más cercanos
    k_indices = None
    
    # Paso 3: Obtener etiquetas de los K vecinos
    k_nearest_labels = None
    
    # Paso 4: Votar por la clase más común
    prediction = None
    ### YOUR CODE ENDS HERE ###
    
    return prediction

# Prueba tu implementación
X_train_ex = np.array([[1, 2], [2, 3], [3, 1], [6, 5], [7, 7], [8, 6]])
y_train_ex = np.array([0, 0, 0, 1, 1, 1])
x_test_ex = np.array([3, 3])

resultado = knn_predict(x_test_ex, X_train_ex, y_train_ex, k=3)
print(f"Predicción para {x_test_ex}: Clase {resultado}")

# Verificar con test automático
verificar_knn = test_ejercicio_2_knn_predict()
verificar_knn(knn_predict)

<a name='ex02'></a>
### Ejercicio 2: Implementar Predicción KNN

Implementa una función que prediga la clase de un punto usando KNN.

**Instrucciones:**
- Usa la función `compute_distances` del ejercicio anterior
- Encuentra los K vecinos más cercanos usando `np.argsort`
- Retorna la clase más común usando votación por mayoría
- Puedes usar `Counter` de collections

In [ ]:
# Comparar K = 1, 5, 15
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, k_val in enumerate([1, 5, 15]):
    knn_compare = KNNClassifier(k=k_val)
    knn_compare.fit(X_train, y_train)
    test_acc = knn_compare.score(X_test, y_test)
    
    # Crear grid
    x_min, x_max = X_train[:, 0].min() - 1, X_train[:, 0].max() + 1
    y_min, y_max = X_train[:, 1].min() - 1, X_train[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    
    Z = knn_compare.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Graficar
    axes[idx].contourf(xx, yy, Z, alpha=0.4, cmap='RdBu')
    axes[idx].scatter(X_train[y_train==0, 0], X_train[y_train==0, 1], 
                     c='red', edgecolors='k', s=50, alpha=0.7)
    axes[idx].scatter(X_train[y_train==1, 0], X_train[y_train==1, 1], 
                     c='blue', edgecolors='k', s=50, alpha=0.7)
    axes[idx].set_xlabel('Feature 1')
    axes[idx].set_ylabel('Feature 2')
    axes[idx].set_title(f'K = {k_val} (Test Acc: {test_acc:.3f})')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Observaciones:")
print("  • K=1: Frontera muy irregular (overfitting)")
print("  • K=5: Balance entre flexibilidad y suavidad")
print("  • K=15: Frontera muy suave (puede tener underfitting)")

### 5.2 - Comparar Fronteras de Decisión con Diferentes K

Veamos cómo cambia la frontera de decisión con diferentes valores de K:

In [ ]:
# Probar diferentes valores de K
k_values = range(1, 21)
train_scores = []
test_scores = []

for k in k_values:
    knn_temp = KNNClassifier(k=k)
    knn_temp.fit(X_train, y_train)
    train_scores.append(knn_temp.score(X_train, y_train))
    test_scores.append(knn_temp.score(X_test, y_test))

# Visualizar
plt.figure(figsize=(12, 5))

# Subplot 1: Accuracy vs K
plt.subplot(1, 2, 1)
plt.plot(k_values, train_scores, 'o-', label='Train', linewidth=2, markersize=6)
plt.plot(k_values, test_scores, 'o-', label='Test', linewidth=2, markersize=6)
plt.xlabel('Valor de K')
plt.ylabel('Accuracy')
plt.title('Accuracy vs K')
plt.legend()
plt.grid(True, alpha=0.3)

# Subplot 2: Gap entre Train y Test
plt.subplot(1, 2, 2)
gap = np.array(train_scores) - np.array(test_scores)
plt.plot(k_values, gap, 'o-', color='purple', linewidth=2, markersize=6)
plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
plt.xlabel('Valor de K')
plt.ylabel('Train - Test Accuracy')
plt.title('Overfitting Gap vs K')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_k = k_values[np.argmax(test_scores)]
print(f"\n🎯 Mejor K: {best_k}")
print(f"   Test Accuracy: {max(test_scores):.4f}")
print(f"   Train Accuracy: {train_scores[best_k-1]:.4f}")

<a name='5'></a>
## 5 - Selección del Hiperparámetro K

El valor de $K$ es crítico para el rendimiento de KNN. Vamos a encontrar el valor óptimo usando validación.

### 5.1 - Impacto de K en el Rendimiento

Probamos diferentes valores de K y observamos su efecto:

In [ ]:
# GRADED FUNCTION: compute_distances

def compute_distances(x, X_train):
    """
    Calcula distancias euclidianas entre un punto y todos los puntos de entrenamiento.
    
    Parámetros
    ----------
    x : ndarray
        Punto de consulta, forma (n_features,)
    X_train : ndarray
        Matriz de puntos de entrenamiento, forma (n_samples, n_features)
    
    Retorna
    -------
    ndarray
        Array de distancias, forma (n_samples,)
    
    Ejemplo
    -------
    >>> x = np.array([1, 2])
    >>> X_train = np.array([[1, 2], [3, 4], [5, 6]])
    >>> compute_distances(x, X_train)
    array([0.        , 2.82842712, 5.65685425])
    """
    
    ### YOUR CODE STARTS HERE ###
    distances = None
    ### YOUR CODE ENDS HERE ###
    
    return distances

# Prueba tu implementación
x_test = np.array([0, 0])
X_train_test = np.array([[1, 0], [0, 1], [3, 4]])

resultado = compute_distances(x_test, X_train_test)
print(f"Distancias desde {x_test} a cada punto de entrenamiento:")
print(resultado)

# Verificar con test automático
verificar_distancias = test_ejercicio_1_distancia()
verificar_distancias(compute_distances)

<a name='ex01'></a>
### Ejercicio 1: Implementar Distancia Euclidiana

Implementa una función vectorizada para calcular distancias euclidianas entre un punto y múltiples puntos.

**Instrucciones:**
- Usa operaciones vectorizadas de NumPy
- La función debe calcular distancias entre `x` y todos los puntos en `X_train` simultáneamente
- Retorna un array de distancias